# NOAA VIIRS Night Lights — Porto Alegre

Pipeline: GEE ImageCollection → annual mean composite → COG → visual + value tiles.

**Source:** `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG` (avg_rad band)  
**Outputs:** `output/poa_nightlights_cog.tif`, `output/tiles_visual/`, `output/tiles_values/`, `output/nightlights_metadata.json`

**Requires:** `earthengine-api`, GDAL

## Step 1. GEE — Export annual mean composite

In [2]:
import ee

ee.Initialize(project='citycatalyst')

In [3]:
# Porto Alegre bounding box [minLon, minLat, maxLon, maxLat]
# Expanded to avoid cutting off municipality boundary (especially east/south)
poa = ee.Geometry.Rectangle([-51.40, -30.42, -50.82, -29.88])

# VIIRS DNB monthly — annual mean composite for 2024
year = "2024"
nightlights = (
    ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
    .filterBounds(poa)
    .filterDate(f"{year}-01-01", f"{int(year)+1}-01-01")
    .select("avg_rad")
    .mean()
)

# Clip and reproject to Web Mercator (~464m native, use 500m for export)
img = nightlights.clip(poa).reproject(crs="EPSG:3857", scale=500)

task = ee.batch.Export.image.toDrive(
    image=img,
    description=f"poa_viirs_nightlights_{year}",
    folder="gee_exports",
    fileNamePrefix=f"poa_viirs_nightlights_{year}",
    region=poa,
    scale=500,
    crs="EPSG:3857",
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

task.start()
print("Started Drive export:", task.id)
print("Download the GeoTIFF from Drive to data/poa_viirs_nightlights_2024.tif before running Step 2.")

Started Drive export: MTLO5SPYJUOGN2AUQST5MAKI
Download the GeoTIFF from Drive to data/poa_viirs_nightlights_2024.tif before running Step 2.


In [4]:
import time

def wait_for_task(task):
    """Poll task status until complete or failed."""
    while task.active():
        status = task.status()
        state = status.get("state", "UNKNOWN")
        print(f"  Status: {state}")
        time.sleep(10)
    status = task.status()
    if status.get("state") == "COMPLETED":
        print("Export finished successfully.")
    else:
        print("Export ended with status:", status)

wait_for_task(task)

  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
Export finished successfully.


## Step 2. GDAL — COG and tiles

Place the downloaded GeoTIFF in `data/poa_viirs_nightlights_2024.tif`, then run the cells below.

In [1]:
import subprocess
import json
from pathlib import Path

# Paths (run from releases/2024/)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
IN_TIF = DATA_DIR / "poa_viirs_nightlights_2024.tif"
OUT_COG = OUTPUT_DIR / "poa_nightlights_cog.tif"
OUT_COLORIZED = OUTPUT_DIR / "poa_nightlights_colorized.tif"
OUT_VALUE_ENCODED = OUTPUT_DIR / "poa_nightlights_value_encoded.tif"
TILES_VISUAL = OUTPUT_DIR / "tiles_visual"
TILES_VALUES = OUTPUT_DIR / "tiles_values"
COLORS_VISUAL = DATA_DIR / "nightlights_visual_colors.txt"
COLORS_VALUE = DATA_DIR / "nightlights_value_encoding_colors.txt"
METADATA_JSON = OUTPUT_DIR / "nightlights_metadata.json"

NIGHTLIGHTS_SCALE = 0.1
NIGHTLIGHTS_OFFSET = 0.0
NIGHTLIGHTS_UNIT = "nW/cm²/sr"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not IN_TIF.exists():
    raise FileNotFoundError(f"Download GeoTIFF from Drive first. Expected: {IN_TIF}")

In [2]:
# Already in EPSG:3857 from GEE; convert to COG
subprocess.run([
    "gdal_translate", str(IN_TIF), str(OUT_COG),
    "-of", "COG",
    "-co", "COMPRESS=DEFLATE",
    "-co", "OVERVIEWS=AUTO",
    "-co", "RESAMPLING=AVERAGE"
], check=True)
print("Created COG:", OUT_COG)

Input file size is 130, 140
0...10...20...30...40...50...60...70...80...90...100 - done.
Created COG: output/poa_nightlights_cog.tif


In [3]:
# Colorize for visual tiles
subprocess.run([
    "gdaldem", "color-relief", str(OUT_COG), str(COLORS_VISUAL), str(OUT_COLORIZED)
], check=True)
print("Colorized for display:", OUT_COLORIZED)

0...10...20...30...40...50...60...70...80...90...100 - done.
Colorized for display: output/poa_nightlights_colorized.tif


In [4]:
# Generate visual XYZ tiles
TILES_VISUAL.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(OUT_COLORIZED), str(TILES_VISUAL)
], check=True)
print("Visual tiles written to:", TILES_VISUAL)

Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:08.
0

Generating Overview Tiles:


...10...20...30...40...50...60...70...80...90...100 - done.
Visual tiles written to: output/tiles_visual


In [9]:
# Value tiles (Terrain RGB) for hover lookup
subprocess.run([
    "gdaldem", "color-relief", str(OUT_COG), str(COLORS_VALUE), str(OUT_VALUE_ENCODED)
], check=True)

TILES_VALUES.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(OUT_VALUE_ENCODED), str(TILES_VALUES)
], check=True)
print("Value tiles written to:", TILES_VALUES)

0...10...20...30...40...50...60...70...80...90...100 - done.


Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0.

Generating Overview Tiles:


..10...20...30...40...50...60...70...80...90...100 - done.
Value tiles written to: output/tiles_values


In [10]:
# Write metadata for frontend
metadata = {
    "layer": "avg_rad",
    "description": "Average DNB radiance (night lights)",
    "time_period": "2024",
    "encoding": {
        "type": "terrain_rgb",
        "scale": NIGHTLIGHTS_SCALE,
        "offset": NIGHTLIGHTS_OFFSET,
        "unit": NIGHTLIGHTS_UNIT,
        "decode_formula": "value = (R * 256 * 256 + G * 256 + B) * scale + offset"
    }
}
with open(METADATA_JSON, "w") as f:
    json.dump(metadata, f, indent=2)
print("Metadata written to:", METADATA_JSON)

Metadata written to: output/nightlights_metadata.json
